# Learned packet omission — Kaggle pilot
Enable **Internet** and a **GPU**. Run cells in order. This trains an expected-cost sender selector and a receiver flow with a reference DCT codec. No modern-codec or mobile-performance advantage is assumed.


In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path("/kaggle/working/dual-flow-packets")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/Ram21275/Packet_flow_Exp.git", str(repo)], check=True)
%cd /kaggle/working/dual-flow-packets
%pip install -q -e ".[test]"


In [ ]:
import torch
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
!python -m pytest -q


## 1. Cache a pilot subset
Streams public [LibriSpeech](https://huggingface.co/datasets/openslr/librispeech_asr) without an HF token. Limits are **utterances**, not hours. Use a new data directory when changing limits. To use attached extracted audio, add `--local-root /kaggle/input/YOUR-DATASET/LibriSpeech`.


In [ ]:
from pathlib import Path
if not Path("/kaggle/working/packets_data/manifest.json").exists():
    subprocess.run([sys.executable, "-m", "packets_exp.cli", "prepare", "--data", "/kaggle/working/packets_data", "--train-limit", "1000", "--valid-limit", "100", "--test-limit", "100"], check=True)


## 2. Train receiver flow
Save notebook outputs before leaving the Kaggle session. Resume with `--resume .../receiver/last.pt`; `--steps` means total optimizer steps.


In [ ]:
!python -m packets_exp.cli train --stage receiver --data /kaggle/working/packets_data --out /kaggle/working/packets_outputs/receiver --steps 5000


## 3. Generate sender targets
Enumerates all five one-frame omission choices using the frozen receiver. Training split only.


In [ ]:
!python -m packets_exp.cli teacher --data /kaggle/working/packets_data --receiver /kaggle/working/packets_outputs/receiver/best.pt --teacher /kaggle/working/packets_outputs/teacher_expected_v2.pt --teacher-windows 20000 --teacher-noise-draws 8


## 4. Train expected-cost sender selector


In [ ]:
!python -m packets_exp.cli train --stage sender --data /kaggle/working/packets_data --teacher /kaggle/working/packets_outputs/teacher_expected_v2.pt --out /kaggle/working/packets_outputs/sender_v2 --steps 3000


## 5. Validate clean and burst-loss channels
Validation first; freeze choices before explicitly evaluating `--split test`.


In [ ]:
!python -m packets_exp.cli evaluate --data /kaggle/working/packets_data --receiver /kaggle/working/packets_outputs/receiver/best.pt --sender /kaggle/working/packets_outputs/sender_v2/best.pt --out /kaggle/working/packets_outputs/valid_clean_v2 --split valid --eval-limit 20


In [ ]:
!python -m packets_exp.cli evaluate --data /kaggle/working/packets_data --receiver /kaggle/working/packets_outputs/receiver/best.pt --sender /kaggle/working/packets_outputs/sender_v2/best.pt --out /kaggle/working/packets_outputs/valid_burst_v2 --split valid --eval-limit 20 --loss .03 --burst


In [ ]:
import pandas as pd
from IPython.display import Audio, display
root = Path("/kaggle/working/packets_outputs/valid_clean_v2")
display(pd.read_csv(root / "summary.csv"))
for wav in sorted(root.glob("learned*wav"))[:3]:
    print(wav.name)
    display(Audio(filename=str(wav)))


## Preserve results
Use Kaggle **Save Version** with outputs. The archive below contains checkpoints, metrics, and listening examples; it does not contain the audio training cache.


In [ ]:
import shutil
shutil.make_archive("/kaggle/working/packets_results", "zip", "/kaggle/working/packets_outputs")
